In [2]:
import os


# Force PyTorch to map sm_90 kernels to your Blackwell GPU
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"

# Keep your existing configs...
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Helps reduce PyTorch memory fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import re
import time
from pathlib import Path
from typing import Optional

import torch
import transformers
import vllm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

print("Python executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("CUDA visible device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)
print("transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} ->", torch.cuda.get_device_name(i))
else:
    raise RuntimeError("CUDA is not available. You are not in a GPU pod/session.")

Python executable: /home/amn024/private/151B_SP26_Competition/.venv/bin/python
CUDA available: True
CUDA visible device count: 1
CUDA version: 12.1
Torch version: 2.5.1+cu121
transformers: 5.9.0
vLLM: 0.7.3
cuda:0 -> NVIDIA A30


In [3]:
print("=== STARTING HARDWARE FUNCTIONALITY CHECK ===")

try:
    # 1. Initialize random 1000x1000 floating-point matrices directly inside the GPU VRAM.
    # This verifies that PyTorch can successfully allocate tensor memory on the Blackwell architecture.
    print("Allocating test matrices on GPU (cuda:0)...")
    matrix_a = torch.randn(1000, 1000, device="cuda")
    matrix_b = torch.randn(1000, 1000, device="cuda")

    # 2. Perform a heavy matrix multiplication (GEMM operations).
    # This forces the NVIDIA hardware driver to compile and execute raw CUDA kernels,
    # proving that the sm_120 hardware is successfully processing code targeting sm_90.
    print("Executing matrix multiplication CUDA kernels...")
    result_matrix = torch.matmul(matrix_a, matrix_b)

    # 3. Synchronize the CUDA device to ensure operations finish without silent background failures.
    torch.cuda.synchronize()
    
    print("\n[SUCCESS] Pipeline is 100% operational!")
    print(f"-> Verified: CUDA is executing operations successfully.")
    print(f"-> Output Tensor Shape: {result_matrix.shape}")
    print("-> Status: You can safely ignore the architecture warning. Your GPU is active and ready.")

except Exception as error:
    print("\n[FAILURE] Hardware check failed. See error details below:")
    print(str(error))

print("=============================================")

=== STARTING HARDWARE FUNCTIONALITY CHECK ===
Allocating test matrices on GPU (cuda:0)...


Executing matrix multiplication CUDA kernels...

[SUCCESS] Pipeline is 100% operational!
-> Verified: CUDA is executing operations successfully.
-> Output Tensor Shape: torch.Size([1000, 1000])
-> Status: You can safely ignore the architecture warning. Your GPU is active and ready.


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

DATA_PATH = "data/public.jsonl"

RUN_NAME = "k=5test"
# # K=1 baseline (sampling, no voting)
# RUN_NAME = "prompt_v2_sc_k1_50";  n=1

# # K=3 self-consistency
# RUN_NAME = "prompt_v2_sc_k3_50";  n=3

# # K=5 self-consistency
# RUN_NAME = "prompt_v2_sc_k5_50";  n=5
OUTPUT_PATH = f"results/{RUN_NAME}.jsonl"

# Conservative first. After it works, raise this to 8192.
MAX_TOKENS = 16384 
# qwen say suse 81k too much for A30 try 32k

# Start with 10. After model loads + scores correctly, change to 50.
EVAL_LIMIT = 50

print("MODEL_ID:", MODEL_ID)
print("DATA_PATH:", DATA_PATH)
print("RUN_NAME:", RUN_NAME)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("MAX_TOKENS:", MAX_TOKENS)
print("EVAL_LIMIT:", EVAL_LIMIT)

MODEL_ID: Qwen/Qwen3-4B-Thinking-2507
DATA_PATH: data/public.jsonl
RUN_NAME: k=5test
OUTPUT_PATH: results/k=5test.jsonl
MAX_TOKENS: 16384
EVAL_LIMIT: 50


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data_path = Path(DATA_PATH)
assert data_path.exists(), f"Cannot find {DATA_PATH}. Run this notebook from the competition repo root."

data = [json.loads(line) for line in open(data_path, encoding="utf-8")]

if EVAL_LIMIT is None:
    eval_data = data
else:
    eval_data = data[:EVAL_LIMIT]

n_mcq_all  = sum(bool(d.get("options")) for d in data)
n_free_all = sum(not d.get("options") for d in data)

n_mcq_eval  = sum(bool(d.get("options")) for d in eval_data)
n_free_eval = sum(not d.get("options") for d in eval_data)

print(f"Loaded {len(data)} total questions  ({n_mcq_all} MCQ, {n_free_all} free-form)")
print(f"Evaluating {len(eval_data)} questions ({n_mcq_eval} MCQ, {n_free_eval} free-form)")

# Preview one MCQ and one free-form item from eval_data
mcq_sample  = next(d for d in eval_data if d.get("options"))
free_sample = next(d for d in eval_data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1500])
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1500])


Loaded 1126 total questions  (375 MCQ, 751 free-form)
Evaluating 50 questions (13 MCQ, 37 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [6]:
# Goal: force a final boxed answer while avoiding endless reasoning loops.

#Using Quen optimization
SYSTEM_PROMPT_FREEFORM = """Please reason step by step, and put your final answer within \\boxed{}.
If the problem has multiple [ANS] blanks, put the answers in order, separated by commas inside a single \\boxed{}.""".strip()
SYSTEM_PROMPT_MCQ = """Please reason step by step, and put your final answer within \\boxed{}.
Please show your choice in the answer field with only the choice letter, e.g., "answer": "C".""".strip()

#SYSTEM_PROMPT_FREEFORM = """
#Please reason step by step, and put your final answer within \\boxed{}.

#Formatting rules:
#1. After thorough verification, your absolute final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the clean, final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas inside the single box.
#4. Do not use words like "approximately" unless the problem explicitly asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#Please reason step by step, and put your final answer within \\boxed{}.

#Formatting rules:
#1. The absolute final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be exactly one capital letter representing the choice (e.g., A, B, C, D, or E).
#3. Do not put the full option text or anything else inside \\boxed{}.
#""".strip()


#SYSTEM_PROMPT_FREEFORM = """
#You are an expert mathematician. Solve this problem using a highly detailed, step-by-step Chain of Thought. 

#Break the problem down into logical sub-tasks. At the end of each major step, rigorously validate your reasoning and arithmetic to ensure no calculation or conceptual errors have occurred. If you detect an inconsistency, backtrack and correct it immediately. 

#Formatting rules:
#1. After thorough verification, your absolute final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the clean, final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas inside the single box.
#4. Do not use words like "approximately" unless the problem explicitly asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#You are an expert mathematician. Use a rigorous Chain of Thought approach to solve this multiple-choice problem.

#First, read and analyze the problem independently without looking at the choices. Derive your result step by step, and validate each stage of your deduction and arithmetic. Once your independent derivation is fully verified, compare your final result against the provided options. If your result does not match any option, re-examine your assumptions and backtrack immediately.

#Formatting rules:
#1. The absolute final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be exactly one capital letter representing the choice (e.g., A, B, C, D, or E).
#3. Do not put the full option text or anything else inside \\boxed{}.
#""".strip()

#SYSTEM_PROMPT_FREEFORM = """
#You are a careful math solver.
#
#Solve the problem step by step, but keep the reasoning concise.
#Do not stop before giving the final answer.

#Formatting rules:
#1. The final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas.
#4. Do not use words like "approximately" unless the problem asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#You are a careful math solver.

#Solve the multiple-choice problem step by step, but keep the reasoning concise.
#Compare your result to the answer choices.

#Formatting rules:
#1. The final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be one capital letter such as A, B, C, D, or E.
#3. Do not put the full option text inside \\boxed{}.
#""".strip()
#'''
# SYSTEM_PROMPT_FREEFORM = (
#     "You are an expert mathematician. Solve this problem using a detailed Chain of Thought. "
#     "Break the problem into logical sub-tasks. At the end of each step, validate your reasoning "
#     "to ensure no calculation or conceptual errors have occurred. "
#     "If you find an inconsistency, backtrack and correct it immediately. "
#     "After thorough verification, put your final answer inside \boxed{}. "
#     "If there are multiple sub-answers, separate them by commas inside the \boxed{}, e.g. \boxed{3, 7}."
# )
# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. Use a Chain of Thought approach to solve this "
#     "multiple-choice problem. First, solve the problem independently without looking at the "
#     "choices. Then, compare your derived result against the provided options. "
#     "Validate each step of your deduction. If your result does not match any option, "
#     "re-examine your assumptions. Output ONLY the capital letter of the single best "
#     "answer inside \boxed{}, e.g. \boxed{C}."
# )


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for one competition item."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{label}. {str(option).strip()}"
            for label, option in zip(labels, options)
        )

        user_prompt = f"""
Problem:
{question}

Answer choices:
{opts_text}

Solve the problem and end with the required boxed letter.
""".strip()

        return SYSTEM_PROMPT_MCQ, user_prompt

    user_prompt = f"""
Problem:
{question}

Solve the problem and end with the required boxed answer.
""".strip()

    return SYSTEM_PROMPT_FREEFORM, user_prompt


# Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print("=" * 80)
#     print(label)
#     print("SYSTEM PROMPT:")
#     print(sys_p)
#     print("\nUSER PROMPT PREVIEW:")
#     print(usr_p[:1000])


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
# ── Load tokenizer + patch Qwen tokenizer compatibility ──────────────────────
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer

if not hasattr(Qwen2Tokenizer, "all_special_tokens_extended"):
    print("Patching Qwen2Tokenizer.all_special_tokens_extended ...")

    @property
    def all_special_tokens_extended(self):
        return list(self.all_special_tokens)

    Qwen2Tokenizer.all_special_tokens_extended = all_special_tokens_extended
else:
    print("Qwen2Tokenizer already has all_special_tokens_extended.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer class:", tokenizer.__class__)
print("Has all_special_tokens_extended:", hasattr(tokenizer, "all_special_tokens_extended"))

# ── Load vLLM model ───────────────────────────────────────────────────────────
'''
vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.75,
    max_model_len=32768,
    max_num_seqs=8,
    max_num_batched_tokens=32768,
    enable_prefix_caching=True,
)
'''
#vllm version 2 optimized for the current a30 gpu 
vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    
    # 1. MEMORY OPTIMIZATION
    gpu_memory_utilization=0.92,    # Push to 92% (Leaves ~1.9GB for system)
    
    # 2. CONTEXT OPTIMIZATION 
    max_model_len=32768,            # 32k is risky on 24GB with BF16; 16k is a safe "fast" zone
    
    # 3. THROUGHPUT OPTIMIZATION
    max_num_seqs=16,                # Increase to handle more questions in parallel
    max_num_batched_tokens=32768,   # Allow larger batches of tokens to be processed
    
    # 4. PREFILL SPEED FIX (The most important one for you)
    enable_chunked_prefill=True,    # This breaks the 4.93 toks/s bottleneck
    
    # 5. CACHING
    enable_prefix_caching=True,     # Keeps your system prompt in memory
)
# Greedy decoding for prompt experiments.
#sampling_params_sc = SamplingParams(
#    max_tokens=MAX_TOKENS,
#    temperature=0,
#    top_p=0.95,
#    n=1,                          # K=1, K=3, K=5
#    repetition_penalty=1.0,
#)

#optimized for thinking models
sampling_params_sc = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,          # Official Qwen3-Thinking production baseline
    top_p=0.95,               # Official Qwen3-Thinking production baseline
    top_k=20,
    n=1,                      # K=1 configuration
    presence_penalty=1.0,     # Essential insurance against internal repetition deadlocks
    repetition_penalty=1.0,
)

print("Model loaded.")

Patching Qwen2Tokenizer.all_special_tokens_extended ...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:03<00:00,  1.62it/s]

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:03<00:00,  1.51it/s]

INFO 05-25 14:15:03 model_runner.py:1562] Graph capturing finished in 3 secs, took 0.12 GiB


INFO 05-25 14:15:03 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 14.32 seconds


Model loaded.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [8]:
# ── Generate K samples per question ───────────────────
def format_chat_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,   # Force Qwen3 thinking mode (<think>...</think>)
    )  #BIG OPTIMIZATION FORCE QWEN TO ALWAYS THINK 

prompts = [format_chat_prompt(item) for item in eval_data]

print(f"Built {len(prompts)} prompts. K={sampling_params_sc.n}")
print(f"Generating {len(prompts) * sampling_params_sc.n} total samples...")

outputs = vllm_model.generate(prompts, sampling_params=sampling_params_sc)

# Keep ALL K samples per question, plus token-level diagnostics.
per_question_raw = []
for out in outputs:
    samples = []
    for o in out.outputs:
        samples.append({
            "text": o.text.strip(),
            "n_tokens": len(o.token_ids),
            "finish_reason": o.finish_reason,   # "stop" or "length" (=truncated)
        })
    per_question_raw.append(samples)

assert len(per_question_raw) == len(eval_data)

K = len(per_question_raw[0])
print(f"\nGeneration complete. K={K}")
print(f"Sample 0, response 0 preview:\n{per_question_raw[0][0]['text'][:400]}")
print(f"\nFinish reasons (first question): {[s['finish_reason'] for s in per_question_raw[0]]}")

Built 50 prompts. K=1
Generating 50 total samples...


Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   2%|▏         | 1/50 [00:21<17:13, 21.09s/it, est. speed input: 4.13 toks/s, output: 40.73 toks/s]

Processed prompts:   4%|▍         | 2/50 [00:22<07:48,  9.76s/it, est. speed input: 8.81 toks/s, output: 77.95 toks/s]

Processed prompts:   6%|▌         | 3/50 [01:07<20:15, 25.87s/it, est. speed input: 9.53 toks/s, output: 58.62 toks/s]

...

Processed prompts:  96%|█████████▌| 48/50 [41:36<04:57, 148.73s/it, est. speed input: 4.03 toks/s, output: 93.56 toks/s]

Processed prompts:  98%|█████████▊| 49/50 [41:38<01:44, 104.87s/it, est. speed input: 4.10 toks/s, output: 98.53 toks/s]

Processed prompts: 100%|██████████| 50/50 [44:03<00:00, 116.87s/it, est. speed input: 3.97 toks/s, output: 99.33 toks/s]

Processed prompts: 100%|██████████| 50/50 [44:03<00:00, 52.87s/it, est. speed input: 3.97 toks/s, output: 99.33 toks/s] 


Generation complete. K=1
Sample 0, response 0 preview:
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few positive even whole numbers are to make sure I know what we're dealing with. The first one is 2, right? Then 4, 6, 8, and so on. Wait, positive 

Finish reasons (first question): ['stop']


In [12]:
def extract_boxed(text: str):
    """
    Extract the last \\boxed{...} content.
    This handles nested braces like \\boxed{\\frac{1}{2}}, unlike a simple regex.
    """
    marker = r"\boxed{"
    start = text.rfind(marker)
    if start == -1:
        return None

    i = start + len(marker)
    depth = 1
    chars = []

    while i < len(text):
        ch = text[i]

        if ch == "{":
            depth += 1
            chars.append(ch)
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(chars).strip()
            chars.append(ch)
        else:
            chars.append(ch)

        i += 1

    return None


# for i in range(min(5, len(responses))):
#     print("=" * 80)
#     print("id:", eval_data[i].get("id"))
#     print("boxed:", extract_boxed(responses[i]))
#     print("response length:", len(responses[i]))
#     print("tail:")
#     print(responses[i][-1000:])


In [13]:
import re

def extract_letter(text: str) -> str:
    """
    Backup helper to find a capital letter (A-E) in the text 
    if the boxed extraction fails or is empty.
    """
    if not text:
        return ""
        
    # 1. Look for common patterns like "The answer is A" or "Choice: B"
    patterns = [
        r"answer is ([A-E])",
        r"answer is: ([A-E])",
        r"answer is \(([A-E])\)",
        r"Choice ([A-E])",
        r"Option ([A-E])",
    ]
    
    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
            
    # 2. Last resort: check the very end of the text for any standalone A-E
    # (Often models end with "Therefore, the answer is B.")
    last_bit = text[-50:].upper()
    m = re.search(r"\b([A-E])\b", last_bit)
    if m:
        return m.group(1).upper()
        
    return ""

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [14]:
# ── Vote + score + record diagnostics ─────────────────
def majority_vote(boxed_answers):
    """Return (voted_answer, status). Status: 'majority' | 'tie_first' | 'all_none'."""
    valid = [b for b in boxed_answers if b is not None]
    if not valid:
        return None, "all_none"
    counts = {}
    for b in valid:
        counts[b] = counts.get(b, 0) + 1
    max_count = max(counts.values())
    winners = {b for b, c in counts.items() if c == max_count}
    if len(winners) == 1:
        return next(iter(winners)), "majority"
    # Tie: deterministic — first occurrence wins
    for b in valid:
        if b in winners:
            return b, "tie_first"

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, samples in tqdm(zip(eval_data, per_question_raw), total=len(eval_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold = item.get("answer", None)

    sample_texts = [s["text"] for s in samples]
    sample_boxed = [extract_boxed(t) for t in sample_texts]

    if K == 1:
        voted, vote_status = sample_boxed[0], "k1"
    else:
        voted, vote_status = majority_vote(sample_boxed)

    # Pick representative trace (the one whose boxed matches the vote)
    if voted is not None:
        rep_idx = next((i for i, b in enumerate(sample_boxed) if b == voted), 0)
    else:
        rep_idx = 0
    rep_text = sample_texts[rep_idx]

    # Score against the VOTED answer (not just sample 0)
    if gold is None:
        correct = None
    elif is_mcq:
        if voted is not None:
            m = re.search(r"\b([A-Z])\b", voted.strip().upper())
            pred_letter = m.group(1) if m else extract_letter(rep_text)
        else:
            pred_letter = extract_letter(rep_text)
        correct = (pred_letter == str(gold).strip().upper())
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=rep_text,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    # Diagnostics across the K samples
    has_boxed_per  = [b is not None for b in sample_boxed]
    truncated_per  = [s["finish_reason"] == "length" for s in samples]
    n_tokens_per   = [s["n_tokens"] for s in samples]

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "K": K,
        "samples_boxed": sample_boxed,
        "voted": voted,
        "vote_status": vote_status,
        "rep_response": rep_text,
        "correct": correct,
        "any_has_boxed":  any(has_boxed_per),
        "all_have_boxed": all(has_boxed_per),
        "any_truncated":  any(truncated_per),
        "all_truncated":  all(truncated_per),
        "tokens_per_sample": n_tokens_per,
        "max_tokens_used": max(n_tokens_per),
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/50 [00:00<?, ?it/s]

Scoring complete. 50 results.


## 8. Summary

Print accuracy broken down by question type.

In [15]:
scored_results = [r for r in results if r["correct"] is not None]
mcq_res  = [r for r in scored_results if r["is_mcq"]]
free_res = [r for r in scored_results if not r["is_mcq"]]

def acc(subset):
    return sum(bool(r["correct"]) for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 60)
print("EVALUATION RESULTS")
print("RUN_NAME:", RUN_NAME)
print("=" * 60)
print(f"  MCQ        : {sum(bool(r['correct']) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(bool(r['correct']) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(bool(r['correct']) for r in scored_results):4d} / {len(scored_results):4d}  ({acc(scored_results):.2f}%)")
print("=" * 60)


EVALUATION RESULTS
RUN_NAME: k=5test
  MCQ        :    8 /   13  (61.54%)
  Free-form  :   18 /   37  (48.65%)
  Overall    :   26 /   50  (52.00%)


In [16]:
# ── Formatting diagnostics (Table 11 in milestone report) ────────────────────
def format_diagnostics(results):
    n = len(results)
    K = results[0]["K"] if results else 0
    has_any   = sum(1 for r in results if r["any_has_boxed"])
    has_all   = sum(1 for r in results if r["all_have_boxed"])
    miss_all  = sum(1 for r in results if not r["any_has_boxed"])
    trunc_any = sum(1 for r in results if r["any_truncated"])
    trunc_all = sum(1 for r in results if r["all_truncated"])
    ties      = sum(1 for r in results if r.get("vote_status") == "tie_first")
    none_vote = sum(1 for r in results if r.get("vote_status") == "all_none")
    avg_tok   = sum(sum(r["tokens_per_sample"]) / len(r["tokens_per_sample"]) for r in results) / n
    pct = lambda x: f"{x}/{n} ({x/n*100:.1f}%)"
    return {
        "RUN_NAME": RUN_NAME,
        "n": n, "K": K,
        "Has Boxed (any sample)":  pct(has_any),
        "Has Boxed (all samples)": pct(has_all),
        "Missing Boxed (all)":     pct(miss_all),
        "Truncated (any sample)":  pct(trunc_any),
        "Truncated (all samples)": pct(trunc_all),
        "Vote ties (K>1 only)":    pct(ties)     if K > 1 else "n/a",
        "All-None votes":          pct(none_vote) if K > 1 else "n/a",
        "Avg tokens/sample":       round(avg_tok, 1),
    }

diag = format_diagnostics(results)
print("=" * 70)
print("FORMATTING DIAGNOSTICS")
print("=" * 70)
for k, v in diag.items():
    print(f"  {k:30s} : {v}")
print("=" * 70)

FORMATTING DIAGNOSTICS
  RUN_NAME                       : k=5test
  n                              : 50
  K                              : 1
  Has Boxed (any sample)         : 47/50 (94.0%)
  Has Boxed (all samples)        : 47/50 (94.0%)
  Missing Boxed (all)            : 3/50 (6.0%)
  Truncated (any sample)         : 2/50 (4.0%)
  Truncated (all samples)        : 2/50 (4.0%)
  Vote ties (K>1 only)           : n/a
  All-None votes                 : n/a
  Avg tokens/sample              : 5251.2


In [ ]:
print("len(data):", len(data))
print("len(eval_data):", len(eval_data))
print("len(prompts):", len(prompts))
print("len(responses):", len(responses))
print("len(results):", len(results))

# Show wrong examples to diagnose prompt failures.
wrong = [r for r in results if r["correct"] is False]

print("wrong count:", len(wrong))

for r in wrong[:5]:
    print("=" * 100)
    print("id:", r["id"], "is_mcq:", r["is_mcq"], "gold:", r["gold"], "boxed:", r["boxed"])
    print("response tail:")
    print(r["response"][-1200:])


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!